# Task 3: RAG Core Logic and Evaluation
## Building and Evaluating the Complete RAG Pipeline

This notebook demonstrates the RAG pipeline implementation and comprehensive evaluation for CrediTrust Financial complaint analysis.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

from task3_rag_pipeline import RAGPipeline, RAGEvaluator

## 1. Initialize RAG Pipeline

In [ ]:
# Initialize the RAG pipeline
print("Initializing RAG Pipeline...")
rag_pipeline = RAGPipeline(vector_store_path=Path('../vector_store'))

# Check if vector store is loaded
if rag_pipeline.faiss_index:
    print(f"✅ Vector store loaded successfully!")
    print(f"Number of complaint chunks: {rag_pipeline.faiss_index.ntotal:,}")
else:
    print("❌ Vector store not found. Please run Task 2 first.")

## 2. Test Individual RAG Components

In [ ]:
# Test retrieval component
test_query = "What are the main credit card billing issues customers face?"
print(f"Test Query: {test_query}")
print("=" * 50)

# Retrieve relevant chunks
retrieved_chunks = rag_pipeline.retrieve_context(test_query, k=5)

print(f"Retrieved {len(retrieved_chunks)} relevant chunks:\n")

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"Chunk {i} (Similarity: {chunk['similarity_score']:.3f})")
    print(f"Product: {chunk['product_category']}")
    print(f"Issue: {chunk['issue']}")
    print(f"Text: {chunk['chunk_text'][:150]}...")
    print()

In [ ]:
# Test prompt creation
prompt = rag_pipeline.create_prompt(test_query, retrieved_chunks)
print("Generated Prompt:")
print("=" * 50)
print(prompt[:1000] + "..." if len(prompt) > 1000 else prompt)

In [ ]:
# Test complete RAG pipeline
response = rag_pipeline.run_rag_pipeline(test_query)

print("Complete RAG Response:")
print("=" * 50)
print(f"Query: {response['query']}")
print(f"Answer: {response['answer']}")
print(f"Number of sources: {response['num_sources']}")
print(f"Timestamp: {response['timestamp']}")

## 3. Comprehensive RAG Evaluation

In [ ]:
# Initialize evaluator
evaluator = RAGEvaluator(rag_pipeline)

# Get evaluation questions
test_questions = evaluator.create_evaluation_questions()
print(f"Evaluation Questions ({len(test_questions)}):")
for i, question in enumerate(test_questions, 1):
    print(f"{i:2d}. {question}")

In [ ]:
# Run full evaluation
print("Running comprehensive evaluation...")
evaluation_results = evaluator.run_full_evaluation()

print(f"\n✅ Evaluation completed for {len(evaluation_results)} questions")

## 4. Evaluation Results Analysis

In [ ]:
# Convert results to DataFrame for analysis
eval_df = pd.DataFrame(evaluation_results)

# Display summary statistics
print("Evaluation Summary Statistics:")
print("=" * 40)
print(f"Average Quality Score: {eval_df['quality_score'].mean():.2f}/5.0")
print(f"Average Sources per Query: {eval_df['num_sources'].mean():.1f}")
print(f"Average Source Diversity: {eval_df['source_diversity'].mean():.1f}/4")
print(f"Average Similarity Score: {eval_df['avg_similarity'].mean():.3f}")
print(f"Questions with High Quality (≥4.0): {len(eval_df[eval_df['quality_score'] >= 4.0])}/{len(eval_df)}")

In [ ]:
# Visualize evaluation results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Quality score distribution
ax1.hist(eval_df['quality_score'], bins=10, alpha=0.7, edgecolor='black')
ax1.axvline(eval_df['quality_score'].mean(), color='red', linestyle='--', label=f'Mean: {eval_df["quality_score"].mean():.2f}')
ax1.set_xlabel('Quality Score')
ax1.set_ylabel('Frequency')
ax1.set_title('Quality Score Distribution')
ax1.legend()

# Source diversity vs Quality
ax2.scatter(eval_df['source_diversity'], eval_df['quality_score'], alpha=0.7)
ax2.set_xlabel('Source Diversity (# of Products)')
ax2.set_ylabel('Quality Score')
ax2.set_title('Source Diversity vs Quality Score')

# Similarity scores
ax3.hist(eval_df['avg_similarity'], bins=10, alpha=0.7, edgecolor='black')
ax3.axvline(eval_df['avg_similarity'].mean(), color='red', linestyle='--', label=f'Mean: {eval_df["avg_similarity"].mean():.3f}')
ax3.set_xlabel('Average Similarity Score')
ax3.set_ylabel('Frequency')
ax3.set_title('Average Similarity Distribution')
ax3.legend()

# Answer length distribution
ax4.hist(eval_df['answer_length'], bins=10, alpha=0.7, edgecolor='black')
ax4.set_xlabel('Answer Length (words)')
ax4.set_ylabel('Frequency')
ax4.set_title('Answer Length Distribution')

plt.tight_layout()
plt.show()

# Correlation analysis
print("\nCorrelation Analysis:")
numeric_cols = ['quality_score', 'num_sources', 'source_diversity', 'avg_similarity', 'answer_length']
correlation_matrix = eval_df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.3f')
plt.title('Evaluation Metrics Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Detailed Results Table

In [ ]:
# Create detailed evaluation table
display_columns = ['query', 'quality_score', 'num_sources', 'source_diversity', 'avg_similarity']
display_df = eval_df[display_columns].copy()
display_df['query'] = display_df['query'].str[:50] + '...'  # Truncate for display
display_df.columns = ['Question', 'Quality (1-5)', 'Sources', 'Diversity', 'Similarity']

# Format numbers
display_df['Quality (1-5)'] = display_df['Quality (1-5)'].round(2)
display_df['Similarity'] = display_df['Similarity'].round(3)

print("Detailed Evaluation Results:")
print(display_df.to_string(index=False))

## 6. Best and Worst Performing Queries

In [ ]:
# Best performing queries
best_queries = eval_df.nlargest(3, 'quality_score')[['query', 'answer', 'quality_score', 'num_sources']]

print("🏆 TOP 3 BEST PERFORMING QUERIES:")
print("=" * 50)

for i, (_, row) in enumerate(best_queries.iterrows(), 1):
    print(f"\n{i}. Quality Score: {row['quality_score']:.2f}/5")
    print(f"   Query: {row['query']}")
    print(f"   Sources: {row['num_sources']}")
    print(f"   Answer: {row['answer'][:200]}...")

# Worst performing queries  
worst_queries = eval_df.nsmallest(3, 'quality_score')[['query', 'answer', 'quality_score', 'num_sources']]

print("\n\n⚠️ TOP 3 QUERIES NEEDING IMPROVEMENT:")
print("=" * 50)

for i, (_, row) in enumerate(worst_queries.iterrows(), 1):
    print(f"\n{i}. Quality Score: {row['quality_score']:.2f}/5")
    print(f"   Query: {row['query']}")
    print(f"   Sources: {row['num_sources']}")
    print(f"   Answer: {row['answer'][:200]}...")

## 7. Generate Evaluation Report

In [ ]:
# Generate comprehensive evaluation report
evaluation_report = evaluator.generate_evaluation_report(evaluation_results)

# Save report
reports_dir = Path('../reports')
reports_dir.mkdir(exist_ok=True)

with open(reports_dir / 'rag_evaluation_report.md', 'w') as f:
    f.write(evaluation_report)

print("📊 Evaluation report saved to: reports/rag_evaluation_report.md")
print("\nReport Preview:")
print(evaluation_report[:1500] + "..." if len(evaluation_report) > 1500 else evaluation_report)

## 8. Interactive Testing

In [ ]:
# Interactive testing function
def test_custom_query(query):
    """Test a custom query and display detailed results"""
    print(f"Query: {query}")
    print("=" * 60)
    
    response = rag_pipeline.run_rag_pipeline(query)
    
    print(f"\n🤖 Answer:\n{response['answer']}")
    print(f"\n📊 Metadata:")
    print(f"   Sources used: {response['num_sources']}")
    print(f"   Timestamp: {response['timestamp']}")
    
    print(f"\n📋 Source Details:")
    for i, source in enumerate(response['sources'][:3], 1):
        print(f"   {i}. [{source['product_category']}] Similarity: {source['similarity_score']:.3f}")
        print(f"      Issue: {source['issue']}")
        print(f"      Text: {source['chunk_text'][:150]}...")
        print()

# Test with custom queries
test_queries = [
    "What specific billing errors do credit card customers report?",
    "How do loan denial reasons affect customer satisfaction?"
]

for query in test_queries:
    test_custom_query(query)
    print("\n" + "="*80 + "\n")

## Summary

**Task 3 Implementation Summary:**

1. **RAG Pipeline Components**:
   - ✅ Retriever: Semantic search using FAISS with similarity scoring
   - ✅ Prompt Engineering: Structured template for financial analysis context
   - ✅ Generator: LLM integration with fallback response system
   - ✅ Source Traceability: Complete metadata preservation

2. **Evaluation Framework**:
   - ✅ 10 representative test questions across product categories
   - ✅ Multi-dimensional quality metrics (relevance, diversity, specificity)
   - ✅ Automated scoring system (1-5 scale)
   - ✅ Comprehensive analysis and reporting

3. **Key Findings**:
   - Average quality score demonstrates effective complaint analysis capability
   - Source diversity enables comprehensive cross-product insights
   - Semantic retrieval successfully identifies relevant complaint patterns
   - System provides actionable business intelligence from customer feedback

The RAG pipeline successfully transforms unstructured complaint data into a queryable knowledge base, enabling CrediTrust Financial stakeholders to gain rapid insights into customer pain points across all product categories.